# Day 3 - Beginner track
## Choosing a model, and being able to prove the choice

**MIPT International Autumn School 2026 | 9 September | 60 minutes**

This morning's lecture gave you a decision procedure: the shape of the data, how much of it
there is, whether you must explain it, and what you can afford. Four constraints, none of
them fashion.

This notebook makes you use that procedure on the dataset you already know, and then defend
the result. On Friday somebody will ask you why you picked the model you picked. "It scored
highest" is not an answer if the difference is smaller than your error bars.

### What you will have at the end

1. Five models trained on identical data, compared honestly.
2. Error bars on every score, so you can say which differences are real.
3. A tuned model, tuned over three parameters and not thirty.
4. One sentence naming the method you will use for your project, and why.

Parts 1 to 4 are the core. Part 5 is the project handover and must not be skipped: it is the
thing the afternoon depends on.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

import sklearn
print("scikit-learn:", sklearn.__version__)

SEED = 42
np.random.seed(SEED)

---
# Part 0 - The data, cleaned the way you cleaned it on Monday

Nothing new here. This is Monday's deterministic cleaning and Tuesday's split, in one cell,
so that the rest of the notebook compares models rather than repeating data work.

Read it anyway. If a line surprises you, that is worth two minutes now rather than a surprise
on Friday.

In [ ]:
df = pd.read_csv("data/car_listings.csv")

# deterministic fixes only: nothing here uses a statistic of the whole column
for col in ["brand", "fuel_type", "transmission", "body_type", "city", "service_history"]:
    df[col] = df[col].str.strip()
df["fuel_type"] = df["fuel_type"].str.lower()
df["city"] = df["city"].str.lower()

df.loc[df["mileage_km"] < 0, "mileage_km"] = np.nan
df.loc[df["model_year"] < 1990, "model_year"] = np.nan

big = df["mileage_km"] > 500_000
df.loc[big, "mileage_km"] = df.loc[big, "mileage_km"] / 10

df = df.drop_duplicates(subset="listing_id").reset_index(drop=True)
df = df.dropna(subset=["price_eur"]).reset_index(drop=True)

# days_on_market is the leak. It is not a feature, it is the answer wearing a hat.
FEATURES = ["brand", "model_year", "mileage_km", "engine_litres", "fuel_type",
            "transmission", "body_type", "owners", "service_history",
            "accident_reported", "city"]
TARGET = "price_eur"

X = df[FEATURES]
y = df[TARGET]
print(X.shape, "rows and features |", "median price:", int(y.median()), "EUR")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED)

print("train:", X_train.shape[0], " test:", X_test.shape[0])
print("The test set is not touched again until Part 4. Not once.")

---
# Part 1 - One harness, five models

The single most common way to compare models dishonestly is to give them slightly different
data. One got the scaled features, another got the raw ones, a third was fitted before a bug
was fixed. The comparison then measures the bugs.

The fix is a harness: one preprocessing definition, one cross-validation scheme, one scoring
function, and every model goes through it unchanged.

We predict `log(price)` and report the error back in euros. Tuesday showed why: the price
distribution has a long right tail, and squared error on the raw price lets a handful of
expensive cars decide everything.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error

NUMERIC = ["model_year", "mileage_km", "engine_litres", "owners", "accident_reported"]
CATEGORICAL = ["brand", "fuel_type", "transmission", "body_type", "service_history", "city"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)


def evaluate(name, model, results):
    """Cross-validate on the training set only. Report MAE in euros, with a standard deviation."""
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    # score on log target, then convert each fold's error back to euros
    scores = cross_val_score(pipe, X_train, np.log(y_train), cv=cv,
                             scoring="neg_mean_absolute_error")
    log_mae = -scores
    # a MAE of e on log(price) is roughly a relative error of (exp(e) - 1)
    rel = np.exp(log_mae) - 1
    results.append({
        "model": name,
        "rel_error_%": round(100 * rel.mean(), 2),
        "std_%": round(100 * rel.std(), 2),
    })
    return results

### The five candidates

The lecture listed five models you will actually use. Here they are, in the order the lecture
gave them. Linear regression goes first because it is the thing every other model has to beat.

Two of the five are left for you. Fill them in with the defaults the lecture recommended:
tune nothing yet, that is Part 3.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

results = []
evaluate("linear regression", LinearRegression(), results)
evaluate("ridge (alpha=1)", Ridge(alpha=1.0), results)
evaluate("decision tree (depth 8)", DecisionTreeRegressor(max_depth=8, random_state=SEED), results)

# TODO: add a random forest. 300 trees, max_depth=None, random_state=SEED, n_jobs=-1


# TODO: add a gradient boosting model. HistGradientBoostingRegressor with default settings
#       and random_state=SEED


table = pd.DataFrame(results).set_index("model")
table

Look at the column of standard deviations before you look at the column of scores.

That number is the spread across the five folds. It is the amount by which your estimate
would move if you had happened to split the data differently, which you very nearly did.

---
# Part 2 - Which differences are real

Two models differ meaningfully only if the gap between them is large compared to the
uncertainty in each. A rough and serviceable rule for five folds: **treat a gap as real when
it exceeds two standard deviations of the better model.** It is not a hypothesis test, but it
stops the single most common mistake of the week, which is celebrating a difference of 0.3
percent.

In [ ]:
best = table["rel_error_%"].idxmin()
best_err = table.loc[best, "rel_error_%"]
best_std = table.loc[best, "std_%"]
print(f"best on cross-validation: {best} at {best_err}% (sd {best_std}%)")

# TODO: for every other model, compute its gap to the best model, and print whether that
#       gap is larger than 2 * best_std. One line per model.

Whatever your table says, write the sentence down now, because it is the sentence you will
say on Friday:

> *We chose X. It beat the others by N percent, which is more (or less) than twice the fold
> to fold spread, so we consider the difference real (or we chose X on grounds of speed and
> interpretability, since the scores were indistinguishable).*

Both versions of that sentence are respectable. Only silence is not.

---
# Part 3 - Tuning, in the amount that is worth it

The lecture was specific: two or three parameters, a coarse grid, cross-validated. A team
that spends the afternoon on a hyperparameter search has spent the afternoon on the least
valuable part of the project.

We will tune gradient boosting over exactly three parameters, on a grid you could have
written by hand, and then check whether the tuning bought anything at all.

In [ ]:
from sklearn.model_selection import GridSearchCV

gb_pipe = Pipeline([("prep", preprocess),
                    ("model", HistGradientBoostingRegressor(random_state=SEED))])

# TODO: build a coarse grid over exactly three parameters:
#         model__learning_rate   : 0.05 and 0.1
#         model__max_depth       : 3, 6 and None
#         model__min_samples_leaf: 10 and 40
#       Use the names exactly as written, with the model__ prefix.
grid = {}

search = GridSearchCV(gb_pipe, grid, cv=cv,
                      scoring="neg_mean_absolute_error", n_jobs=-1)
search.fit(X_train, np.log(y_train))

tuned_rel = 100 * (np.exp(-search.best_score_) - 1)
print("best parameters:", search.best_params_)
print(f"tuned cross-validation error: {tuned_rel:.2f}%")

In [ ]:
untuned = table.loc["gradient boosting", "rel_error_%"]
print(f"untuned: {untuned:.2f}%   tuned: {tuned_rel:.2f}%   gained: {untuned - tuned_rel:.2f} points")
print(f"fold to fold spread of the untuned model: {table.loc['gradient boosting', 'std_%']:.2f} points")

Compare the gain to the spread. On this dataset, twelve fits of cross-validated searching
usually buys less than the noise between folds.

That is not an argument against tuning. It is an argument for tuning **last**, after the
question, the features and the honesty of the evaluation are settled, and for stopping as
soon as the gain stops exceeding the noise.

---
# Part 4 - Pick the winner, then spend the test set once

The choice is made on cross-validation. The test set only measures the model you already
chose. If you look at the test score for all five and then pick the best, you have chosen on
the test set, and the number you report afterwards is no longer an estimate of anything.

So: compare the tuned gradient boosting against the best model from Part 1, on
cross-validation, and ship whichever wins. On this dataset the answer is often not the one
students expect, and that is the point of the day.

In [ ]:
candidates = {
    "linear regression": Pipeline([("prep", preprocess), ("model", LinearRegression())]),
    "tuned gradient boosting": search.best_estimator_,
}

cv_scores = {}
for name, pipe in candidates.items():
    s = cross_val_score(pipe, X_train, np.log(y_train), cv=cv,
                        scoring="neg_mean_absolute_error")
    cv_scores[name] = 100 * (np.exp(-s.mean()) - 1)
    print(f"{name:<26} {cv_scores[name]:.2f}%")

chosen = min(cv_scores, key=cv_scores.get)
print()
print("chosen on cross-validation:", chosen)

final = candidates[chosen]
final.fit(X_train, np.log(y_train))

pred = np.exp(final.predict(X_test))
mae = mean_absolute_error(y_test, pred)
mape = 100 * np.mean(np.abs(pred - y_test) / y_test)

print(f"test MAE:  {mae:,.0f} EUR")
print(f"test MAPE: {mape:.1f}%")
print(f"median price in the test set: {y_test.median():,.0f} EUR")

Translate before you move on. A MAE of around nine hundred euros, on a car whose median price
is a little over eight thousand, means the model is typically wrong by roughly a tenth of the
price of the car. State it in those terms on Friday. A dealer does not think in MAE, and
"nine hundred euros out on a typical car" is a sentence they can act on.

---
# Part 5 - Interpretability, and then your project

Your project has to explain itself. Two tools, and they answer different questions.

**Coefficients** of a linear model say: holding everything else fixed, this feature moves the
prediction by this much. They are a statement about the model, they are only a statement about
the world if the model is right.

**Permutation importance** says: if I shuffle this column so it carries no information, how
much worse does the model get. It works for any model, including the ones with no coefficients.

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(final, X_test, np.log(y_test),
                             n_repeats=5, random_state=SEED, n_jobs=-1)

order = np.argsort(imp.importances_mean)[::-1]
for i in order[:8]:
    print(f"{FEATURES[i]:<20} {imp.importances_mean[i]:.4f}  (sd {imp.importances_std[i]:.4f})")

In [ ]:
# TODO: answer in a comment, in one sentence each.
#
# 1. Which feature would hurt the model most if it went missing tomorrow?
#
# 2. Your project question, in one sentence.
#
# 3. The method you will use this afternoon, and which of the four constraints from the
#    lecture decided it (shape of data / how much of it / must you explain it / what you
#    can afford).
#
# 4. The baseline it has to beat, and by how much before you would call the difference real.
#

---
## What to carry into the afternoon

1. **Linear regression is the thing to beat, and it is harder to beat than you expect.**
2. **A difference smaller than twice the fold spread is not a difference.** Say so out loud
   rather than pretending it is a win.
3. **Tune three parameters, coarsely, last.**
4. **The test set is spent once.** Every extra look costs you the right to call it an estimate.
5. **Permutation importance works on any model**, including the one you cannot interpret.

Your afternoon starts with the sentence you wrote in Part 5, question 3. Bring it to the
instructor before you write any project code.